# Data Cleaning — Online Retail II

Clean the merged retail dataset for **Customer Behavior Shift Detection**.

**Steps:**
1. Load raw merged data
2. Drop rows with missing `Customer ID`
3. Drop exact duplicate rows
4. Keep valid purchases (`Quantity > 0`, `Price > 0`)
5. Fix dtypes and add `TotalPrice`
6. Save cleaned data to `data/processed/online_retail_II_full.csv`

## 1. Imports & load

In [5]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/online_retail_II_full.csv")
OUTPUT_PATH = Path("../data/processed/online_retail_II_full.csv")

df = pd.read_csv(RAW_PATH)
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom


## 2. Quick inspection

In [6]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB
None

Missing values:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Exact duplicate rows: 34335


## 3. Drop missing Customer ID

Behavior-shift modeling needs a known customer identity.

In [7]:
before = len(df)
df = df.dropna(subset=["Customer ID"]).copy()
print(f"Removed missing Customer ID: {before - len(df):,}")
print(f"Rows left: {len(df):,}")

Removed missing Customer ID: 243,007
Rows left: 824,364


## 4. Drop exact duplicate rows

In [8]:
before = len(df)
df = df.drop_duplicates(keep="last").copy()
print(f"Removed duplicates: {before - len(df):,}")
print(f"Rows left: {len(df):,}")
print("Remaining exact duplicates:", df.duplicated().sum())

Removed duplicates: 26,479
Rows left: 797,885
Remaining exact duplicates: 0


## 5. Keep valid purchases only

Remove cancellations / invalid lines (`Quantity <= 0` or `Price <= 0`).

In [9]:
before = len(df)
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()
print(f"Removed invalid qty/price rows: {before - len(df):,}")
print(f"Rows left: {len(df):,}")

Removed invalid qty/price rows: 18,460
Rows left: 779,425


## 6. Fix types & add TotalPrice

In [10]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"] = df["Customer ID"].astype("int64")
df["TotalPrice"] = df["Quantity"] * df["Price"]

print(df.dtypes)
df.head()

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
TotalPrice            float64
dtype: object


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 7. Final quality check

In [11]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())
print("Negative/zero Quantity:", (df["Quantity"] <= 0).sum())
print("Negative/zero Price:", (df["Price"] <= 0).sum())
print("Unique customers:", df["Customer ID"].nunique())
print("Date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())

Final shape: (779425, 9)

Missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
dtype: int64

Exact duplicates: 0
Negative/zero Quantity: 0
Negative/zero Price: 0
Unique customers: 5878
Date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


In [12]:
# Remove non-commercial transactions such as manual adjustments and test products

non_commercial_codes = ["M", "ADJUST", "ADJUST2", "TEST001", "TEST002"]

non_commercial_mask = df["StockCode"].isin(non_commercial_codes)

print("Non-commercial rows:", non_commercial_mask.sum())

df = df[~non_commercial_mask].copy()

print("Rows after removing non-commercial transactions:", len(df))

Non-commercial rows: 726
Rows after removing non-commercial transactions: 778699


## 8. Save cleaned dataset

Writes to `data/processed/online_retail_II_full.csv` (raw file stays unchanged).

In [15]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print("Saved cleaned dataset to:", OUTPUT_PATH.resolve())
print("Saved shape:", df.shape)

Saved cleaned dataset to: F:\GSG26_Course\fina_ml_project\data\processed\online_retail_II_full.csv
Saved shape: (778699, 9)


In [16]:
# Verify the written file
check_df = pd.read_csv(OUTPUT_PATH)
print("Reloaded shape:", check_df.shape)
print("Exact duplicates in saved file:", check_df.duplicated().sum())
check_df.head()

Reloaded shape: (778699, 9)
Exact duplicates in saved file: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0
